In [0]:
%run ../utils/adls_auth

In [0]:
%run ../utils/control_table

In [0]:
# Disable deletion vectors for Synapse compatibility
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

In [0]:

import uuid
from datetime import datetime
from pyspark.sql.functions import col, lit, sha2, concat_ws, current_date, current_timestamp,to_date
from delta.tables import DeltaTable


spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

In [0]:
SOURCE_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/zone_lookup_raw"
GOLD_PATH = "abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_location"
HIGH_DATE = "9999-12-31"

In [0]:
source_df = (
    spark.read.option("header", True).csv(SOURCE_PATH)
    .select(
        col("LocationID").cast("int").alias("location_id"),
        col("Borough").alias("borough"),
        col("Zone").alias("zone"),
        col("service_zone"),
    )
)

if not DeltaTable.isDeltaTable(spark, GOLD_PATH):
    

    
    
    initial_df = (
    source_df
    .withColumn("location_key", sha2(concat_ws("_", col("location_id"), lit("2020-01-01")), 256))
    .withColumn("effective_start_date", to_date(lit("2020-01-01")))
    .withColumn("effective_end_date", lit(HIGH_DATE).cast("date"))
    .withColumn("is_current", lit(True))
        )
    initial_df.write.format("delta").mode("overwrite").save(GOLD_PATH)
    print(f"dim_location initial load: {initial_df.count()} rows.")

else:
    dim_table = DeltaTable.forPath(spark, GOLD_PATH)
    current_df = dim_table.toDF().filter(col("is_current") == True)

    changed_df = (
        source_df.alias("src")
        .join(current_df.alias("cur"), col("src.location_id") == col("cur.location_id"), "inner")
        .filter(
            (col("src.borough") != col("cur.borough")) |
            (col("src.zone") != col("cur.zone")) |
            (col("src.service_zone") != col("cur.service_zone"))
        )
        .select("src.*")
    )

    new_df = source_df.join(current_df, "location_id", "left_anti")  

    rows_to_insert = changed_df.unionByName(new_df).withColumn(
        "location_key", sha2(concat_ws("_", col("location_id"), current_timestamp().cast("string")), 256)
    ).withColumn("effective_start_date", current_date()) \
     .withColumn("effective_end_date", lit(HIGH_DATE).cast("date")) \
     .withColumn("is_current", lit(True)) \
     .withColumn("merge_key", lit(None).cast("string"))  # null merge_key forces INSERT below

    rows_to_expire = changed_df.select("location_id").withColumn("merge_key", col("location_id").cast("string"))

    staged = rows_to_insert.unionByName(
        rows_to_expire.withColumn("borough", lit(None).cast("string"))
                      .withColumn("zone", lit(None).cast("string"))
                      .withColumn("service_zone", lit(None).cast("string"))
                      .withColumn("location_key", lit(None).cast("string"))
                      .withColumn("effective_start_date", lit(None).cast("date"))
                      .withColumn("effective_end_date", lit(None).cast("date"))
                      .withColumn("is_current", lit(None).cast("boolean")),
        allowMissingColumns=True,
    )

    (dim_table.alias("target")
        .merge(staged.alias("staged"), "target.location_id = staged.merge_key AND target.is_current = true")
        .whenMatchedUpdate(set={
            "is_current": lit(False),
            "effective_end_date": (current_date() - 1).cast("date"),
        })
        .whenNotMatchedInsert(values={
            "location_id": col("staged.location_id"),
            "borough": col("staged.borough"),
            "zone": col("staged.zone"),
            "service_zone": col("staged.service_zone"),
            "location_key": col("staged.location_key"),
            "effective_start_date": col("staged.effective_start_date"),
            "effective_end_date": col("staged.effective_end_date"),
            "is_current": col("staged.is_current"),
        })
        .execute())

    print(f"dim_location SCD2 merge complete: {changed_df.count()} changed, {new_df.count()} new locations.")